# Grouped Summaries

*Group records by category and interpret counts, totals, and means for each group.*


## Records and Group Keys

Each row is an order. The `team` column repeats because a team can place
several orders. A grouped summary combines those records rather than
treating each row as a different team.


In [1]:
import pandas as pd

orders_df = pd.DataFrame({
    "team": ["A", "A", "B", "B", "A", "C"],
    "product": ["Mouse", "Keyboard", "Mouse", "Monitor", "Monitor", "Keyboard"],
    "sales": [50, 80, 55, 200, 180, 90],
    "quantity": [5, 4, 6, 2, 1, 3],
})

print("Order records:")
print(orders_df)


Order records:
  team   product  sales  quantity
0    A     Mouse     50         5
1    A  Keyboard     80         4
2    B     Mouse     55         6
3    B   Monitor    200         2
4    A   Monitor    180         1
5    C  Keyboard     90         3


Six records describe three teams. Team A has three orders, B has two, and C has one.


## Split, Aggregate, and Combine

`.groupby("team")` identifies groups; selecting numeric columns and
calling `.sum()` produces one summary row per team. Group keys become
index labels by default. `sort=True` puts those keys in a predictable order.


In [2]:
team_totals = orders_df.groupby("team", sort=True)[["sales", "quantity"]].sum()
team_means = orders_df.groupby("team", sort=True)["sales"].mean()

print("Sales and quantities by team:")
print(team_totals)
print("Average order sales by team:")
print(team_means.round(2))


Sales and quantities by team:
      sales  quantity
team                 
A       310        10
B       255         8
C        90         3
Average order sales by team:
team
A    103.33
B    127.50
C     90.00
Name: sales, dtype: float64


Team A has the largest total, 310, but team B has the largest mean, 127.5. A's three orders average about 103.33; C's single order is 90.


## Give Summary Columns Meaningful Names

Named aggregation uses `new_name=("source_column", "operation")`.
`as_index=False` retains the team as a regular column. `size` counts
rows, while `count` would count non-missing values in the selected column.


In [3]:
team_report = orders_df.groupby("team", as_index=False, sort=True).agg(
    order_count=("sales", "size"),
    total_sales=("sales", "sum"),
    mean_sales=("sales", "mean"),
    total_quantity=("quantity", "sum"),
)

print("Named team report:")
print(team_report.round({"mean_sales": 2}))
print("Grouped sales total:", team_report["total_sales"].sum())
print("All sales accounted for:", team_report["total_sales"].sum() == orders_df["sales"].sum())


Named team report:
  team  order_count  total_sales  mean_sales  total_quantity
0    A            3          310      103.33              10
1    B            2          255      127.50               8
2    C            1           90       90.00               3
Grouped sales total: 655
All sales accounted for: True


The team totals add to 655 and quantities add to 21. The equality check is `True`: every order contributes once to the grouped sales total.


Choose a group key and a statistic that describe the records at the intended level of detail. Compare totals with counts or means when group sizes differ.

[Group by: split, apply, combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
